# v12s 訓練 — yolo26**s**-p2 @ Datasets_YOLO26_v5.6

**這一輪只改一個變因：模型尺度 n → s。** 其餘超參數、排程、資料集、
評估流程與 v11.5 **逐字相同**，所以任何差異都只能歸因到容量。

---

## 為什麼現在才試放大模型

到目前為止**模型端的改動全數無效**：

| 嘗試 | 結果 |
| --- | --- |
| v9 六臂消融（W-I-MPDIoU ×2、ADown、StarTripletBlock、imgsz 800） | **零臂通過 2σ 門檻**，且越貴的改動結果越差 |
| v10 → v11 → v11.5（同一批 326 張 v5.5 test） | mAP50 全距 **0.021** = run 間雜訊 |

唯一有效過的手段是**資料端的標註重整**。

但有兩件事變了，讓「試更大的模型」第一次**變得可評估**：

1. **量尺修好了。** v11.5 讓九類的 pooled ±2SE ≤ 0.10，per-class 雜訊地板也量出來是 ±0.04。
   在此之前，任何 0.02 的差異都無法與雜訊區分。
2. **部署端有了硬指標。** 30 FPS ±5，且有一台實體手機當測試平台 1。
   在此之前沒有延遲預算，「要不要用 s 模型」根本無法討論——現在可以問
   「多花的 3.49× 計算量換到多少精度」。

---

## 所需資料

| 項目 | 內容 |
| --- | --- |
| 資料集 | **`Datasets_YOLO26_v5.6`**（本機 `Datasets/2_處理與切分/v5.6/`）上傳成 Kaggle Dataset |
| 規模 | 9 類，`train 7,264 / valid 401 / test 401` 影像，`18,078 / 868 / 822` 框 |
| 內含 | `data.yaml`、`train/`、`valid/`、`test/`、`_provenance.json` |
| **與 v11.5 用的是同一份** | 不重新建置、不重新上傳。這是「只改一個變因」的前提 |
| 網路 | **Notebook 的 Internet 必須開啟**（要抓 `yolo26s.pt`，不是 `yolo26n.pt`） |

> Step.2 的張數指紋檢查（`valid==401`、`test==401`、`train==7264`）原封不動保留。
> 跑錯資料集會直接停在 Step.2。

---

## 參數

**與 v11.5 逐項相同，只有模型尺度不同：**

| 項目 | v11.5 | **v12s** | 倍率 |
| --- | --- | --- | ---: |
| 架構 | `yolo26-p2` scale **n** | `yolo26-p2` scale **s** | — |
| 參數量 | 2,518,088 | **9,667,624** | **3.84×** |
| GFLOPs @640 | 7.65 | **26.67** | **3.49×** |
| 預訓練權重 | `yolo26n.pt` | **`yolo26s.pt`** | — |
| `epochs` / `patience` | 70 / 0 | **70 / 0** | 不動 |
| 其餘超參數 | — | **完全相同** | — |
| 報告權重 | `last.pt` | **`last.pt`** | 不動 |

（參數量與 GFLOPs 是本機用 `DetectionModel` 實測的，不是文件抄來的。）

```
optimizer=MuSGD   lr0=0.008   lrf=0.01   momentum=0.937   cos_lr=True
warmup_epochs=3   mosaic=0.7  cls=0.8    dfl=1.5   box=8.0
imgsz=640         batch=20    seed=0     cache=ram   close_mosaic=13
```

> **`lr0` 刻意不隨模型放大而調整。** 調了就多一個變因，本輪就不再是純粹的尺度比較。
> 若 s 因為學習率不合而表現不佳，那是**下一輪**要單獨測的東西，不是這一輪要順手改的。

---

## 訓練說明

架構是**原封不動的官方 `yolo26-p2`**（`end2end=True`、`reg_max=1`，NMS-free），
只是把 `scale` 從 `n` 換成 `s`——`scales` 字典裡 n 是 `[0.5, 0.25, 1024]`、
s 是 `[0.5, 0.5, 1024]`，**深度倍率相同，只有寬度從 0.25 變 0.5**。

流程與 v11.5 相同：Step.1 環境 → Step.2 資料集複製與指紋檢查 → Step.3 模型組態 →
**Step.4 架構隔離驗證 ＋ GPU 顯存探測** → Step.5 訓練 → Step.6 壓縮 → Step.7 評估包。

**Step.4 比 v11.5 多一段 GPU 顯存探測**：在真正的 batch size 上做一次
前向＋反向，把峰值顯存印出來。理由見下面的風險 3——與其跑了資料複製之後才 OOM，
不如在 30 秒內就知道。

Step.7 一樣是 **2 個權重 × 2 個 split 共四次評估**，另外追加與 v11.5 的逐類對照。

建議用 **Save & Run All**，全程無人看顧。

---

## 這一輪要回答的問題與判準（預先登記）

> 先寫死判準，是為了避免事後看到數字再找理由。

**問題**：在這份資料上，模型容量是不是瓶頸？

| | |
| --- | --- |
| **主判準** | v12s 與 v11.5 在**同一批 v5.6 test（401 張）**上的 mAP50 差距 |
| **基準值** | v11.5 `last.pt` @ v5.6 test = **0.80959**（取自 v11.5 notebook 內 Step.7 的 `summary.json`） |
| **達標線** | **> +0.03**，即 v12s test mAP50 **≥ 0.83959** |
| **為什麼是 0.03** | per-class 雜訊地板 ±0.04、三版 run 間全距 0.021。低於 0.03 的差距無法與雜訊區分 |

**結論怎麼寫**：

- **≥ +0.03** → 容量確實是瓶頸之一。接著要看 benchmark 的延遲代價，決定值不值得
- **< +0.03** → **容量不是瓶頸**。結論是把預算移回資料端（人工工作包 A：`Thrips_Damage` 的框定義）
- **明顯變差** → 過擬合。這也是有用的結論，代表 2,353 張原始影像撐不起 9.7 M 參數

> **基準值要用對。** v11.5 有兩組數字：notebook 內 Step.7 的 `eval/summary.json`
> （test mAP50 **0.80959**）與事後本機重跑的 `local_eval/overview_metrics.json`
> （0.80950）。兩者差 0.0001，但 valid 差了 0.006。
> **本輪比的是 notebook 內那一組**，因為 v12s 的 Step.7 跑的是同一段程式碼——
> 同一條量測路徑才叫 like-for-like。

**FPS 不是本輪的通過條件**（使用者已選擇「若過不了就放寬 FPS 目標」），
但報告仍要把 v12s 的 FPS 與 30 的差距列出來供最後取捨。

---

## 預期成果

| # | 預期 | 判準 |
| --- | --- | --- |
| 1 | **達標線大概率過不了** | 這是誠實的預期，不是唱衰：v9 六臂零臂通過，v10/v11/v11.5 全距只有雜訊等級。若連 3.84× 參數都推不動 0.03，那訊號很明確 |
| 2 | **最佳輪次比 ep53 更早** | v11.5 的 `val/cls` 在 ep39 就見底，而 raw train 只有 2,353 張。參數放大 3.84× 在同樣的資料上，過擬合會更早來 |
| 3 | `Thrips_Damage` **不會因為模型變大而變好** | 它的成因是**框定義不一致**（valid 0.653 / test 0.451，差 0.202），那是標註問題，不是容量問題。若它真的大幅改善，反而該懷疑是雜訊 |
| 4 | 平台期 σ 與 v11.5 同量級（0.0045 附近） | 評估集沒變 |

---

## ⚠️ 三個風險

### 1. 時間可能吃緊

s-p2 是 **3.49×** 的 GFLOPs。v11.5 實測 **157 s/epoch**。

| 項目 | 預估 |
| --- | ---: |
| 訓練 70 輪 | **7.6 – 9.1 小時**（390–470 s/epoch） |
| 資料集複製 ＋ Step.7 四次評估 | 約 0.5 小時 |
| **合計** | **約 8.1 – 9.6 小時** |

估區間是抓的，不是量的：epoch 時間不會純粹跟著 FLOPs 走，資料管線是固定成本。
Kaggle 單場 12 h，`DEADLINE_HOURS = 10.5` 是安全閥。
**真的不夠就會在 10.5 h 停下並留 `resume_from.pt`**，用 RESUME 續跑。

### 2. `last.pt` 的正當性要重新檢視

v11.5 用 `last.pt` 當主要結果，理由是實測 `best` 與 `last` 差距全部 ≤ 0.004、
方向還相反——沒有可辨識的差異。**s 模型不見得也是這樣**：若它更早過擬合，
`last.pt`（ep70）可能明顯劣於平台期。

Step.7 會把 `last` 對平台期的差距印出來。**若 `last` 低於平台期超過 2σ，
報告裡必須點出來，不能沿用 v11.5 的說法。**

### 3. `batch=20` 可能 OOM

s-p2 的寬度是 n 的 2 倍，而 P2 偵測頭的 stride-4 特徵圖是 160×160——
這是整個網路最吃顯存的地方。

**若必須調降 batch，那是一個額外的變因**，本輪就不再是純粹的尺度比較，
報告裡要明說。Step.4 的顯存探測會在訓練開始前就給出訊號。

> 探測是 fp32、沒有 optimizer state、沒有 EMA。
> **探測 OOM ⇒ 訓練一定 OOM**（可信）；**探測過了 ⇒ 訓練大概率過**（不保證）。


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 唯一需要編輯的 cell
# ══════════════════════════════════════════════════════════════════════
RUN      = "v5.6_v12s"    # 訓練輸出目錄名稱
EPOCHS   = 70             # 與 v11.5 相同，不動
PATIENCE = 0              # 0 = 關閉早停。valid 因此不參與任何決策，可與 test 併計

# ── 本輪唯一的變因 ────────────────────────────────────────────────────
SCALE      = "s"          # v11.5 是 "n"。scales: n=[0.5,0.25,1024]  s=[0.5,0.5,1024]
PRETRAINED = "yolo26s.pt" # 必須跟著 SCALE 換，否則轉移率會掉一大截

# Datasets_YOLO26_v5.6（本機 Datasets/2_處理與切分/v5.6）上傳成 Kaggle Dataset 後的路徑
# **與 v11.5 用同一份**，不重新上傳。
DATASET_SRC = "/kaggle/input/datasets-yolo26-v5-6"

# s-p2 估 7.6–9.1 h。DEADLINE_HOURS 是安全閥：超時會乾淨停止並留下 resume_from.pt。
STOP_AFTER_EPOCHS = None
DEADLINE_HOURS    = 10.5

# ── 超參數：與 v11.5 逐字相同，維持可對照 ──────────────────────────────
# lr0 刻意不隨模型放大而調整——調了就多一個變因，本輪就不是純粹的尺度比較。
HYPERS = dict(
    optimizer="MuSGD", lr0=0.008, lrf=0.01, momentum=0.937, cos_lr=True,
    warmup_epochs=3.0, mosaic=0.7, cls=0.8, dfl=1.5, box=8.0, imgsz=640,
    batch=20, seed=0, cache="ram", workers=4, plots=True, device=0,
)

CLOSE_MOSAIC = max(1, round(30 * EPOCHS / 160))   # = 13，與 v11.5 相同

# v10 實測的 test ±2SE 與當時的 test 影像數，用來投影本輪的 ±2SE（Step.7 會算）
SE_BASELINE = {
    "Oily_Spot": (0.022, 24), "Canker": (0.061, 24), "Sooty_Mold": (0.030, 32),
    "Black_Spot": (0.043, 25), "Scale_Insect": (0.082, 25),
    "Citrus_Leaf_Miner": (0.210, 20), "Thrips": (0.170, 39),
    "Aphid": (0.048, 76), "Thrips_Damage": (0.161, 21),
}

# ══════════════════════════════════════════════════════════════════════
# 預先登記的判準——先寫死，避免事後看到數字再找理由
#
# 基準值取自 v11.5 notebook 內 Step.7 的 eval/summary.json，**不是**事後本機重跑的
# local_eval（那份 test 差 0.0001、valid 差 0.006）。同一條量測路徑才叫 like-for-like。
# ══════════════════════════════════════════════════════════════════════
BASELINE = {
    "run": "v11.5", "weight": "last.pt", "dataset": "v5.6",
    "val":  {"mAP50": 0.82899, "mAP50_95": 0.63580},
    "test": {"mAP50": 0.80959, "mAP50_95": 0.60691},
    "plateau16": {"mAP50": (0.83041, 0.00256), "mAP50_95": (0.63190, 0.00450)},
    "per_class_ap50": {   # (valid, test)
        "Oily_Spot": (0.97055, 0.94458), "Canker": (0.92823, 0.86765),
        "Sooty_Mold": (0.99500, 0.99500), "Black_Spot": (0.97984, 0.95340),
        "Scale_Insect": (0.63025, 0.67041), "Citrus_Leaf_Miner": (0.71571, 0.67645),
        "Thrips": (0.70006, 0.77946), "Aphid": (0.88801, 0.94785),
        "Thrips_Damage": (0.65323, 0.45148),
    },
}
# 達標線：test mAP50 至少要比 v11.5 高 0.03。
# 為什麼是 0.03——per-class 雜訊地板 ±0.04、三版 run 間全距 0.021，低於這個數字
# 的差異無法與雜訊區分。
PASS_DELTA = 0.03
PASS_MARK = round(BASELINE["test"]["mAP50"] + PASS_DELTA, 5)

assert PATIENCE == 0, "與 v11.5 對照的前提就是 patience=0；要改請先讀第一個 cell"
assert SCALE == "s" and PRETRAINED == "yolo26s.pt", "SCALE 與預訓練權重必須成對"
print(f"▷ RUN={RUN}  scale={SCALE}  pretrained={PRETRAINED}")
print(f"▷ epochs={EPOCHS}  patience={PATIENCE}（早停已關閉）  close_mosaic={CLOSE_MOSAIC}")
print(f"▷ 達標線  test mAP50 ≥ {PASS_MARK}"
      f"（v11.5 {BASELINE['test']['mAP50']} + {PASS_DELTA}）")


# Step.1 環境

In [ ]:
!nvidia-smi
# 版本釘死：v8 以來的所有數字都在 8.4.121 上取得，換版本會失去可對照性
!pip install -q ultralytics==8.4.121

import ultralytics
assert ultralytics.__version__ == "8.4.121", \
    f"ultralytics 版本不符：{ultralytics.__version__}"
print(f"▷ ultralytics {ultralytics.__version__}")

# Step.2 資料集準備
### 複製到可寫入工作區、校驗完整性、清 BOM 與舊快取、改寫 data.yaml 路徑。

In [ ]:
import os, shutil, sys, yaml

def render_progress_bar(current, total, task_name="檔案同步複製中", bar_length=25):
    percent = (current / total) * 100 if total > 0 else 100.0
    filled = int(bar_length * current // total) if total > 0 else bar_length
    bar = "█" * filled + "░" * (bar_length - filled)
    sys.stdout.write(f"\r▷ 正在執行 [{task_name}] | 進度: [{bar}] {percent:5.1f}% ({current}/{total})")
    sys.stdout.flush()


def copy_and_verify_dataset(src_dir, dst_dir):
    if not os.path.exists(src_dir):
        print(f"▷ 錯誤：找不到來源資料集目錄 {src_dir}")
        return False

    src_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            src_files.append(os.path.relpath(os.path.join(root, file), src_dir))
    total_files = len(src_files)
    print(f"▷ 來源資料集掃描完成，共計 {total_files} 個檔案")

    for idx, rel_path in enumerate(src_files, 1):
        dst_path = os.path.join(dst_dir, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(os.path.join(src_dir, rel_path), dst_path)
        if idx % 200 == 0 or idx == total_files:
            render_progress_bar(idx, total_files)

    print("\n\n▷ 正在檢查複製檔案")
    dst_files_set = set()
    for root, _, files in os.walk(dst_dir):
        for file in files:
            dst_files_set.add(os.path.relpath(os.path.join(root, file), dst_dir))

    missing, corrupted = [], []
    for rel_path in src_files:
        if rel_path not in dst_files_set:
            missing.append(rel_path)
        elif os.path.getsize(os.path.join(src_dir, rel_path)) != \
                os.path.getsize(os.path.join(dst_dir, rel_path)):
            corrupted.append(rel_path)

    print("≡" * 60)
    print("▷ 資料集複製完整性校驗：")
    print(f"  ▶ 來源檔案總數 : {len(src_files)}")
    print(f"  ▶ 目標檔案總數 : {len(dst_files_set)}")
    print(f"  ▶ 遺漏檔案數   : {len(missing)}")
    print(f"  ▶ 損毀/大小不符: {len(corrupted)}")
    ok = not missing and not corrupted
    print("▷ 檢查通過" if ok else f"▷ 檢查失敗  遺漏={missing[:5]}  損毀={corrupted[:5]}")
    print("≡" * 60)
    return ok


DST = "/kaggle/working/datasets-yolo26-v5-6"
DATA_YAML = "/kaggle/working/data.yaml"

assert copy_and_verify_dataset(DATASET_SRC, DST), "資料集複製失敗，不要往下跑"

# data.yaml 改寫成絕對路徑（來源版本用的是相對的 path: .）
with open(os.path.join(DST, "data.yaml"), encoding="utf-8") as f:
    ycfg = yaml.safe_load(f)
ycfg.update(path=DST, train="train/images", val="valid/images", test="test/images")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(ycfg, f, default_flow_style=False, allow_unicode=True)

# v5.5 是 9 類。這裡刻意不寫死數字以外的假設：nc 由 data.yaml 決定，
# 後面的模型組態與驗證全部沿用 ycfg["nc"]。
NC = int(ycfg["nc"])
assert NC == 9, f"nc={NC}，v5.6 應為 9（8 = 舊的 v5r，跑錯資料集了）"
assert ycfg["names"][8] == "Thrips_Damage", f"第 9 類應為 Thrips_Damage，實際是 {ycfg['names'][8]}"

# 這一項是 v11 之後新加的。v5.5 與 v5.6 的 nc 與類別名完全相同，只有評估集大小不同
# （v5.5 是 327/326、v5.6 是 401/401）。v11 有一半的分析時間花在事後確認「到底跑了哪一版」，
# 因為評估腳本裡留了一個寫死的舊字串。拿張數當指紋，跑錯就直接停在這裡。
_n = {sp: len(os.listdir(os.path.join(DST, sp, "images"))) for sp in ("train", "valid", "test")}
print(f"▷ 影像張數 {_n}")
assert _n["valid"] == 401 and _n["test"] == 401, (
    f"valid/test 應為 401/401（v5.6），實際 {_n['valid']}/{_n['test']}。"
    f"327/326 代表上傳的是 v5.5。")
assert _n["train"] == 7264, f"train 應為 7,264（v5.6），實際 {_n['train']}"

# BOM 與舊快取會讓 Ultralytics 的標註解析出錯
bom_fixed = cache_removed = 0
for subdir, _, files in os.walk(DST):
    for file in files:
        path = os.path.join(subdir, file)
        if file.endswith(".cache"):
            os.remove(path)
            cache_removed += 1
        elif file.endswith(".txt") and "labels" in subdir:
            with open(path, "rb") as fh:
                is_bom = fh.read(3) == b"\xef\xbb\xbf"
            if is_bom:
                with open(path, encoding="utf-8-sig") as fh:
                    content = fh.read()
                with open(path, "w", encoding="utf-8") as fh:
                    fh.write(content)
                bom_fixed += 1

print(f"▷ data.yaml → {DATA_YAML}   nc={NC}")
print(f"▷ names = {ycfg['names']}")
print(f"▷ 修正 BOM {bom_fixed} 個、清除快取 {cache_removed} 個")
print("▷ Step.2 完成")

# Step.3 模型組態
### 直接用 ultralytics 內建的 `yolo26-p2.yaml`，只覆寫 `nc`。不做任何模組替換。

In [ ]:
import yaml as _yaml
from pathlib import Path
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import get_flops, get_num_params

# 讀已安裝的官方組態。v9 時代是先讀進來、再依臂別替換層；v10 起一層都不動。
_ULTRA = Path(ultralytics.__file__).parent
cfg = _yaml.safe_load((_ULTRA / "cfg/models/26/yolo26-p2.yaml").read_text(encoding="utf-8"))

cfg["nc"] = NC
cfg["scale"] = SCALE            # ← 本輪唯一的變因（v11.5 是 "n"）
# end2end / reg_max 必須保留：YOLO26 靠這兩個 key 決定走 NMS-free 的 E2EDetectLoss
# （box + cls + l1）還是舊的 DFL(reg_max=16) 路徑。漏掉會靜默變成另一個體系的模型，
# 與 v8 / v9 完全無法對照。官方 yaml 本來就有，這裡只是明示不可被覆寫掉。
cfg.setdefault("end2end", True)
cfg.setdefault("reg_max", 1)

# 檔名必須帶 scale 字母：yaml_model_load 會用檔名覆寫 dict 裡的 scale 鍵，
# 檔名寫錯會落回 scales 字典的第一項（也就是 n）並只印一行 warning——
# 那會讓整輪實驗變成 v11.5 的重跑，而且不容易發現。
YAML_PATH = f"/kaggle/working/yolo26{SCALE}-p2-v12s.yaml"
with open(YAML_PATH, "w", encoding="utf-8") as f:
    _yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

_m = DetectionModel(cfg=cfg, ch=3, nc=NC, verbose=False)
_P, _G = get_num_params(_m), get_flops(_m, imgsz=640)
print(f"▷ v12s({SCALE}): {_P:,} params / {_G:.2f} GFLOPs @640")
print(f"▷ 對 v11.5(n) 的倍率: params {_P / 2_518_088:.2f}x   GFLOPs {_G / 7.65:.2f}x")

# 本機實測過的 s-p2 @ nc=9 就是這兩個數字。對不上代表 scale 沒吃進去，
# 或 ultralytics 換了版本——兩種情況都會讓本輪失去可對照性，直接停。
assert abs(_P - 9_667_624) < 5_000, f"params={_P:,}，s-p2@nc9 應為 9,667,624（是不是 scale 沒生效？）"
assert abs(_G - 26.67) < 0.3, f"GFLOPs={_G:.2f}，s-p2 應為 26.67"
print("▷ 尺度確認：這是 s，不是 n")
del _m

HP = dict(HYPERS)
HP.update(epochs=EPOCHS, patience=PATIENCE, close_mosaic=CLOSE_MOSAIC, name=RUN)
print("▷ 超參數：" + "  ".join(f"{k}={v}" for k, v in sorted(HP.items())))


# Step.4 架構隔離驗證
### 在燒 GPU 時數之前確認：體系正確、可前向、損失有限。

In [ ]:
import math, torch
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
import ultralytics.utils.loss

model = YOLO(YAML_PATH)
core = model.model

# 1. E2E 體系必須與 v8 / v9 一致，否則走的是另一個損失路徑、完全無法對照
det = core.model[-1]
assert getattr(det, "end2end", False), "Detect 不是 end2end：yaml 少了 end2end: True"
assert det.reg_max == 1, f"reg_max={det.reg_max}，應為 1"
assert det.nc == NC and det.nl == 4, f"nc={det.nc} nl={det.nl}"
strides = [int(s) for s in det.stride]
assert strides == [4, 8, 16, 32], f"strides={strides}"
print(f"▷ 1/5 end2end=True, reg_max=1, nc={det.nc}, 偵測頭={det.nl}, strides={strides}")

# 2. 確認是乾淨的官方架構——v12s 不該有任何自訂模組或被 patch 過的損失。
#    （這條在 v9 是「該有的模組要就位」，v10 起反過來：一個都不該有。）
types = {m.type.split(".")[-1] for m in core.model}
for forbidden in ("ADown", "StarTripletBlock"):
    assert forbidden not in types, f"不該出現 {forbidden}，v12s 是原封不動的官方架構，只換 scale"
assert ultralytics.utils.loss.bbox_iou.__name__ == "bbox_iou", \
    f"損失函式被換過了：{ultralytics.utils.loss.bbox_iou.__name__}，v12s 應為內建 CIoU"
print(f"▷ 2/5 無自訂模組、無 loss patch，確認為原封不動的官方 yolo26-p2（scale={SCALE}）")

# 3. 前向
core.eval()
with torch.no_grad():
    core(torch.zeros(1, 3, HP["imgsz"], HP["imgsz"]))
print(f"▷ 3/5 Forward pass ({HP['imgsz']}x{HP['imgsz']}) 成功")

# 4. 完整損失路徑，刻意用微小框貼近本資料集分佈
core.args = get_cfg(overrides={"box": HP["box"], "cls": HP["cls"], "dfl": HP["dfl"]})
core.train()
loss, items = core.loss({
    "img": torch.rand(2, 3, HP["imgsz"], HP["imgsz"]),
    "batch_idx": torch.tensor([0.0, 0.0, 1.0]),
    "cls": torch.tensor([[4.0], [8.0], [1.0]]),      # Scale_Insect / Thrips_Damage / Canker
    "bboxes": torch.tensor([[0.50, 0.50, 0.04, 0.04],
                            [0.22, 0.31, 0.20, 0.18],
                            [0.71, 0.68, 0.03, 0.03]]),
})
vals = ({k: float(v) for k, v in items.items()} if isinstance(items, dict)
        else {k: float(v) for k, v in zip(("box", "cls", "l1"), items.flatten())})
assert torch.isfinite(loss).all() and all(math.isfinite(v) for v in vals.values()), vals
print("▷ 4/5 損失路徑通過   " + "  ".join(f"{k}={v:.4f}" for k, v in vals.items()))

# ══════════════════════════════════════════════════════════════════════
# 5. GPU 顯存探測（v11.5 沒有這一段）
#
# s-p2 的寬度是 n 的 2 倍，而 P2 頭的 stride-4 特徵圖是 160x160——整個網路最吃
# 顯存的地方。與其跑完資料複製、訓練到第一個 batch 才 OOM，不如在這裡花 30 秒。
#
# 這個探測是 fp32、沒有 optimizer state、沒有 EMA、沒有 AMP：
#   探測 OOM  ⇒ 訓練一定 OOM（可信）
#   探測通過  ⇒ 訓練大概率通過（不保證）
# ══════════════════════════════════════════════════════════════════════
if not torch.cuda.is_available():
    print("▷ 5/5 沒有 GPU，跳過顯存探測")
else:
    B, S = HP["batch"], HP["imgsz"]
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    try:
        gm = YOLO(YAML_PATH).model.cuda()
        gm.args = get_cfg(overrides={"box": HP["box"], "cls": HP["cls"], "dfl": HP["dfl"]})
        gm.train()
        gl, _ = gm.loss({
            "img": torch.rand(B, 3, S, S, device="cuda"),
            "batch_idx": torch.arange(B, dtype=torch.float32).repeat_interleave(2).cuda(),
            "cls": torch.randint(0, NC, (B * 2, 1)).float().cuda(),
            "bboxes": torch.tensor([[0.5, 0.5, 0.06, 0.06]] * (B * 2)).cuda(),
        })
        gl.sum().backward()
        peak = torch.cuda.max_memory_allocated() / 2**30
        print(f"▷ 5/5 batch={B} @ {S}px 前向+反向通過   "
              f"峰值顯存 {peak:.2f} / {total:.1f} GiB（{peak / total:.0%}）")
        if peak / total > 0.75:
            print("     ⚠ 已用掉 75% 以上。訓練還要加上 optimizer state 與 EMA，")
            print("       若 Step.5 OOM，把 batch 調降並**在報告裡標明這是額外的變因**。")
        del gm, gl
    except torch.cuda.OutOfMemoryError:
        print(f"▷ 5/5 ✗ batch={B} 在 {total:.1f} GiB 上 OOM。")
        print("     把 HYPERS['batch'] 調降（建議先試 12），重跑本 cell。")
        print("     ⚠ 調降 batch 是**額外的變因**，本輪就不再是純粹的尺度比較，")
        print("       報告裡必須明說。")
        raise
    finally:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

print("\n▷ Step.4 全部通過，可以開始訓練")


# Step.5 訓練

In [ ]:
import shutil, time
from ultralytics import YOLO

model = YOLO(YAML_PATH)
model.load(PRETRAINED)       # 架構與官方一致，轉移率應為 902/902
                             # ← 權重必須跟 SCALE 對應，n 的權重載進 s 會掉一大截


def stop_and_snapshot(trainer):
    """每輪保留可續跑的 checkpoint，並在時數/輪數上限時乾淨停止。

    訓練迴圈結束後一定會執行 final_eval() → strip_optimizer()，把 last.pt / best.pt
    的 epoch 改成 -1 並清掉 optimizer/EMA，那種檔案無法續跑。
    on_fit_epoch_end 的觸發點在 save_model() 之後、跳出迴圈之前，此時 last.pt
    才剛寫好且尚未被 strip。

    備份不設條件：patience 早停時 trainer.stop 在進入這個 callback 之前就已為 True，
    若寫成 `if not trainer.stop` 這段會整個被跳過——那正是 v9 踩過的失效模式。
    本輪 patience=0 不會早停，但這段照留：它同時也是牆鐘超時的退場路徑。
    """
    if trainer.last.exists():
        shutil.copy(trainer.last, trainer.wdir / "resume_from.pt")

    cap = STOP_AFTER_EPOCHS or trainer.epochs      # None → 跑滿，不提前停
    elapsed = (time.time() - trainer.train_time_start) / 3600
    if not trainer.stop and (trainer.epoch + 1 >= cap or elapsed > DEADLINE_HOURS):
        trainer.stop = True
        print(f"\n▷ 停於第 {trainer.epoch + 1} / {trainer.epochs} 輪，已耗時 {elapsed:.2f} h")
        print(f"▷ 續跑用 checkpoint：{trainer.wdir / 'resume_from.pt'}")


model.add_callback("on_fit_epoch_end", stop_and_snapshot)

_cap = STOP_AFTER_EPOCHS or EPOCHS
print(f"▷ 將跑到第 {_cap} / {EPOCHS} 輪"
      + ("" if _cap >= EPOCHS else "  ← STOP_AFTER_EPOCHS 會提前停止，剩餘輪次需用 RESUME.ipynb")
      + f"；牆鐘上限 {DEADLINE_HOURS} h")

results = model.train(data=DATA_YAML, save_period=10, **HP)
print("▷ v12s 訓練完畢")

# Step.6 輸出整理

In [ ]:
import os, shutil

runs_dir = "/kaggle/working/runs"
if os.path.exists(runs_dir):
    print("▷ 正在壓縮訓練輸出")
    shutil.make_archive(f"/kaggle/working/runs_{RUN}", "zip", runs_dir)
    size = os.path.getsize(f"/kaggle/working/runs_{RUN}.zip") / (1024 ** 2)
    print(f"▷ 壓縮成功 /kaggle/working/runs_{RUN}.zip ({size:.2f} MB)")
else:
    print(f"▷ 壓縮失敗：找不到 {runs_dir}")

# Step.7 評估包

2 個權重 × 2 個 split 共四次評估，另加與 v11.5 的逐類對照與判準裁決。


In [ ]:
import csv, json, os, shutil, zipfile
from ultralytics import YOLO


def write_csv(path, fieldnames, rows):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


RUN_DIR = str(model.trainer.save_dir)
OUT = f"/kaggle/working/eval_{RUN}"
os.makedirs(OUT, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════
# 2 個權重 × 2 個 split，共四次評估
#
#   last.pt  ep70 固定輪數跑完的自然終點，valid 完全沒參與任何決策
#            → **valid 與 test 可以合法併計**，這是本輪的主要結果
#   best.pt  由 fitness（在 valid 上算）挑出，帶選擇偏誤
#            → 只作對照，用來量 best-of-N 灌水有多大
#
# plots=True 是必要的，不是為了畫圖：ultralytics 把 confusion_matrix.process_batch
# 包在 `if self.args.plots` 裡（detect/val.py:196），plots=False 會讓混淆矩陣維持全零。
# ══════════════════════════════════════════════════════════════════════
WEIGHTS = {"last": os.path.join(RUN_DIR, "weights", "last.pt"),
           "best": os.path.join(RUN_DIR, "weights", "best.pt")}
evals = {}

for tag, wp in WEIGHTS.items():
    for split in ("val", "test"):
        mm = YOLO(wp).val(data=DATA_YAML, split=split, imgsz=HP["imgsz"],
                          batch=HP["batch"], plots=True)
        nm = mm.names if isinstance(mm.names, dict) else dict(enumerate(mm.names))
        per = {}
        for i, ci in enumerate(mm.box.ap_class_index):
            ci = int(ci)
            per[nm.get(ci, str(ci))] = {
                "precision": round(float(mm.box.p[i]), 5),
                "recall": round(float(mm.box.r[i]), 5),
                "f1": round(float(mm.box.f1[i]), 5),
                "ap50": round(float(mm.box.ap50[i]), 5),
                "ap50_95": round(float(mm.box.ap[i]), 5),
            }
        evals[f"{tag}_{split}"] = {
            "mAP50": round(float(mm.box.map50), 5),
            "mAP50_95": round(float(mm.box.map), 5),
            "precision": round(float(mm.box.mp), 5),
            "recall": round(float(mm.box.mr), 5),
            "per_class": per,
        }
        print(f"▷ {tag}.pt @ {split}   mAP50 {mm.box.map50:.5f}   mAP50-95 {mm.box.map:.5f}")
        if tag == "last" and split == "test":
            m_main, cm_main = mm, mm.confusion_matrix.matrix
            names = nm

# ── 逐輪指標與超參數 ────────────────────────────────────────────────
for fn in ("results.csv", "args.yaml"):
    src = os.path.join(RUN_DIR, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUT, fn))

# ── 每類指標（四種組合各一份）──────────────────────────────────────
COLS = ["class", "precision", "recall", "f1", "ap50", "ap50_95"]
for k, e in evals.items():
    write_csv(os.path.join(OUT, f"per_class_{k}.csv"), COLS,
              [{"class": c, **v} for c, v in e["per_class"].items()])

# ── 混淆矩陣（last.pt @ test；列=預測，欄=真實，最後一列/欄為背景）──
nc = len(names)
labels = [names.get(i, str(i)) for i in range(nc)] + ["background"]
with open(os.path.join(OUT, "confusion_matrix.csv"), "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow([""] + [f"true_{l}" for l in labels])
    for i, lab in enumerate(labels):
        w.writerow([f"pred_{lab}"] + [int(cm_main[i][j]) for j in range(len(labels))])
fp_bg = {labels[i]: int(cm_main[i][nc]) for i in range(nc)}
fn_bg = {labels[i]: int(cm_main[nc][i]) for i in range(nc)}

# ── F1-信心曲線與最佳截斷點（last.pt @ test）────────────────────────
best_conf = None
try:
    x, y, _xl, _yl = m_main.curves_results[1]        # F1-Confidence(B)
    x = [float(v) for v in x]
    mean_f1 = ([sum(col) / len(col) for col in zip(*y)] if hasattr(y[0], "__len__")
               else [float(v) for v in y])
    write_csv(os.path.join(OUT, "f1_conf.csv"), ["conf", "mean_f1"],
              [{"conf": a, "mean_f1": b} for a, b in zip(x, mean_f1)])
    best_conf = round(x[mean_f1.index(max(mean_f1))], 4)
except Exception as e:
    print(f"▷ F1-conf 曲線取用失敗（不影響主要結果）：{type(e).__name__}: {e}")

# ── 平台期統計 ──────────────────────────────────────────────────────
plateau = {}
rcsv = os.path.join(OUT, "results.csv")
if os.path.exists(rcsv):
    with open(rcsv, encoding="utf-8") as f:
        rec = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]
    win = 50 if EPOCHS >= 120 else 16
    tail = rec[-win:]
    for key, col in (("mAP50", "metrics/mAP50(B)"), ("mAP50_95", "metrics/mAP50-95(B)")):
        if rec and col in rec[0]:
            vals = [float(r[col]) for r in tail]
            mean = sum(vals) / len(vals)
            var = sum((v - mean) ** 2 for v in vals) / max(len(vals) - 1, 1)
            plateau[key] = {"window": win, "mean": round(mean, 5),
                            "std": round(var ** 0.5, 5), "best": round(max(vals), 5)}

# ══════════════════════════════════════════════════════════════════════
# 本輪的重點：pooled（valid + test）的逐類 ±2SE
#
# 只有 last.pt 能這樣算——它是固定輪數跑完的終點，valid 沒有參與任何決策。
# ±2SE 由 v10 實測值依 1/sqrt(n) 投影；真值要事後跑 diag_localization.py --bootstrap。
# ══════════════════════════════════════════════════════════════════════
N_EVAL = {"Oily_Spot": (24, 24), "Canker": (24, 24), "Sooty_Mold": (33, 32),
          "Black_Spot": (25, 25), "Scale_Insect": (35, 35),
          "Citrus_Leaf_Miner": (45, 45), "Thrips": (60, 60),
          "Aphid": (75, 76), "Thrips_Damage": (40, 40)}

pooled, worst = {}, 0.0
print(f"\n{'類別':<20}{'valid AP50':>11}{'test AP50':>10}{'差':>8}{'pooled ±2SE':>13}{'':>4}")
for c, (nv, nt) in N_EVAL.items():
    va = evals["last_val"]["per_class"].get(c, {}).get("ap50")
    ta = evals["last_test"]["per_class"].get(c, {}).get("ap50")
    se0, n0 = SE_BASELINE[c]
    se = round(se0 * (n0 / (nv + nt)) ** 0.5, 4)
    worst = max(worst, se)
    gap = None if (va is None or ta is None) else round(ta - va, 4)
    pooled[c] = {"n_valid": nv, "n_test": nt, "ap50_valid": va, "ap50_test": ta,
                 "gap": gap, "se2_pooled": se, "meets_target": se <= 0.10}
    print(f"{c:<20}{(va if va is not None else float('nan')):>11.3f}"
          f"{(ta if ta is not None else float('nan')):>10.3f}"
          f"{(gap if gap is not None else float('nan')):>8.3f}"
          f"{se:>13.3f}{'✓' if se <= 0.10 else '✗':>4}")
print(f"\n▷ 最差 pooled ±2SE = {worst:.3f}   目標 0.10   "
      f"{'★ 達成' if worst <= 0.10 else '✗ 未達成'}")

inflation = None
if plateau.get("mAP50_95"):
    inflation = round(evals["best_val"]["mAP50_95"] - plateau["mAP50_95"]["mean"], 5)
    sd = plateau["mAP50_95"]["std"] or 1e-9
    print(f"▷ best.pt 高出平台期 {inflation:+.5f} = {inflation / sd:.1f}σ"
          f"（v10 2.8σ、v11 2.7σ、v11.5 1.5σ）")

summary = {
    "run": RUN, "scale": SCALE, "epochs": EPOCHS, "patience": PATIENCE, "close_mosaic": CLOSE_MOSAIC,
    "dataset": "v5.6",                      # 由 Step.2 的張數指紋確認過
    "nc": NC, "imgsz": HP["imgsz"],
    "primary_weight": "last.pt",
    "note": ("patience=0 且固定輪數，valid 未參與任何決策，"
             "因此 last.pt 的 valid 與 test 可合法併計；best.pt 僅作對照。"),
    "evals": evals,
    "pooled_2se": pooled,
    "worst_pooled_2se": round(worst, 4),
    "plateau": plateau,
    "best_vs_plateau": inflation,
    "best_f1_conf": best_conf,
    "fp_from_background": fp_bg,
    "fn_to_background": fn_bg,
}
with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# ══════════════════════════════════════════════════════════════════════
# 本輪追加：與 v11.5 的對照，以及預先登記判準的裁決
#
# 只比 last.pt——v11.5 報的是 last.pt，兩邊都是固定輪數跑完的自然終點。
# 拿 v12s 的 best.pt 去比 v11.5 的 last.pt 是灌水，這裡刻意不提供那個數字。
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 74)
print("  v12s(s-p2)  vs  v11.5(n-p2)   —— 同一份 v5.6，同樣的超參數與排程")
print("=" * 74)

delta = {}
for sp, key in (("valid", "last_val"), ("test", "last_test")):
    b = BASELINE["val" if sp == "valid" else "test"]
    for metric in ("mAP50", "mAP50_95"):
        d = round(evals[key][metric] - b[metric], 5)
        delta[f"{sp}_{metric}"] = d
        print(f"  {sp:<6}{metric:<10} v11.5 {b[metric]:.5f}  →  v12s {evals[key][metric]:.5f}"
              f"   {d:+.5f}")

print("\n  逐類 AP50（valid / test），Δ 是 v12s − v11.5")
print(f"  {'類別':<20}{'v11.5 valid':>12}{'v12s valid':>11}{'Δ':>9}"
      f"{'v11.5 test':>12}{'v12s test':>10}{'Δ':>9}")
per_class_delta = {}
for c, (bv, bt) in BASELINE["per_class_ap50"].items():
    nv = evals["last_val"]["per_class"].get(c, {}).get("ap50")
    nt = evals["last_test"]["per_class"].get(c, {}).get("ap50")
    dv = None if nv is None else round(nv - bv, 4)
    dt = None if nt is None else round(nt - bt, 4)
    per_class_delta[c] = {"valid": dv, "test": dt}
    print(f"  {c:<20}{bv:>12.3f}{(nv if nv is not None else float('nan')):>11.3f}"
          f"{(dv if dv is not None else float('nan')):>+9.3f}"
          f"{bt:>12.3f}{(nt if nt is not None else float('nan')):>10.3f}"
          f"{(dt if dt is not None else float('nan')):>+9.3f}")
print("\n  ⚠ per-class 的雜訊地板是 ±0.04（v11 實測：資料完全沒動的五個類別也移動這麼多）。")
print("    上表任何 |Δ| < 0.04 的欄位一律不解讀。")

# ── 預先登記判準的裁決 ────────────────────────────────────────────────
achieved = evals["last_test"]["mAP50"]
d_test = delta["test_mAP50"]
passed = achieved >= PASS_MARK

print("\n" + "-" * 74)
print(f"  預先登記的主判準：v5.6 test mAP50 ≥ {PASS_MARK}"
      f"（v11.5 {BASELINE['test']['mAP50']} + {PASS_DELTA}）")
print(f"  實得 {achieved:.5f}   Δ = {d_test:+.5f}")
if passed:
    print("  ★ 通過 —— 容量確實是瓶頸之一。")
    print("    下一步：看 benchmark 的延遲代價（s-p2 是 3.49x GFLOPs），決定值不值得。")
else:
    print("  ✗ 未通過 —— 在這份資料上，模型容量不是瓶頸。")
    print(f"    3.84x 參數只換到 {d_test:+.4f}，而 run 間雜訊本身就有 0.021。")
    print("    結論：把預算移回資料端（人工工作包 A：Thrips_Damage 的框定義不一致，")
    print("          valid/test 差 0.202，三輪都沒收斂）。")
print("-" * 74)

# ── last.pt 的正當性複查（風險 2）────────────────────────────────────
# v11.5 用 last.pt 當主要結果，前提是 best 與 last 沒有可辨識的差異。
# s 模型若更早過擬合，last.pt(ep70) 可能明顯劣於平台期——那就不能沿用那個說法。
last_vs_plateau = None
if plateau.get("mAP50_95"):
    pm, ps = plateau["mAP50_95"]["mean"], plateau["mAP50_95"]["std"] or 1e-9
    last_vs_plateau = round(evals["last_val"]["mAP50_95"] - pm, 5)
    sigma = last_vs_plateau / ps
    print(f"\n  last.pt @ valid 相對平台期：{last_vs_plateau:+.5f} = {sigma:+.1f}σ"
          f"（平台期 mean {pm:.5f} σ {ps:.5f}）")
    if sigma < -2:
        print("  ⚠ last.pt 低於平台期超過 2σ ——**不能沿用 v11.5「best≈last」的說法**。")
        print("    這是過擬合的訊號。報告裡要改用平台期平均，並說明改動的理由。")
    else:
        print("  ✓ 與平台期沒有可辨識的差異，沿用 last.pt 作為主要結果是成立的。")

summary["comparison_vs_v11_5"] = {
    "baseline": BASELINE,
    "delta": delta,
    "per_class_delta_ap50": per_class_delta,
    "pass_mark": PASS_MARK,
    "pass_delta": PASS_DELTA,
    "achieved_test_mAP50": achieved,
    "passed": bool(passed),
    "last_vs_plateau_mAP50_95": last_vs_plateau,
    "note": ("只比 last.pt。基準值取自 v11.5 notebook 內的 eval/summary.json，"
             "與本輪 Step.7 走同一條量測路徑。per-class 雜訊地板 ±0.04。"),
}
# _P / _G 由 Step.3 定義（那裡也 assert 過它們就是 s-p2 的數字）
summary["model"] = {"scale": SCALE, "pretrained": PRETRAINED,
                    "params": int(_P), "gflops_640": round(float(_G), 2)}
with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

shutil.make_archive(f"/kaggle/working/eval_{RUN}", "zip", OUT)
print(f"\n▷ 評估包（已含對照）→ /kaggle/working/eval_{RUN}.zip")
